# Wisconsin Breast Cancer 数据探索（新版课程适配）

本 Notebook 对应新版课程中的数据理解、基准性能、倾斜数据集和规范数据划分。

学习目标：

1. 理解样本、特征和标签；
2. 查看数据集形状和列名；
3. 明确正类与负类；
4. 计算类别分布和多数类基线；
5. 检查缺失值、重复样本和特征尺度；
6. 分层划分训练集、验证集与测试集；
7. 理解为什么预处理只能在训练集上 fit。


## 1. 导入库

这一部分导入数据处理、绘图和数据划分需要的库。

In [ ]:
# 导入 Path，用于构造项目文件路径。
from pathlib import Path

# 导入 NumPy，用于数值计算。
import numpy as np

# 导入 Pandas，用于表格数据处理。
import pandas as pd

# 导入 Matplotlib，用于数据可视化。
import matplotlib.pyplot as plt

# 导入 scikit-learn 内置乳腺癌二分类数据集。
from sklearn.datasets import load_breast_cancer

# 导入数据集划分函数。
from sklearn.model_selection import train_test_split

# 设置 Pandas 最多显示 100 列，便于观察 30 个特征。
pd.set_option("display.max_columns", 100)

# 设置浮点数默认显示 4 位小数。
pd.set_option("display.float_format", lambda value: f"{value:.4f}")

# 打印环境检查信息。
print("NumPy version:", np.__version__)

# 打印 Pandas 版本。
print("Pandas version:", pd.__version__)

## 2. 加载数据集

scikit-learn 原始标签定义：

```text
0 = malignant
1 = benign
```

为了让“恶性”成为我们重点评价的正类，本项目重新映射：

```text
0 = benign
1 = malignant
```

In [ ]:
# 以 DataFrame 形式加载数据集。
dataset = load_breast_cancer(as_frame=True)

# 复制 30 个输入特征。
X = dataset.data.copy()

# 把原始 malignant=0 映射为新的正类标签 1。
y = (dataset.target == 0).astype(int)

# 给标签 Series 设置清晰名称。
y.name = "is_malignant"

# 创建便于阅读的类别名称映射。
class_names = {0: "benign", 1: "malignant"}

# 打印数据集英文说明中的类别名称。
print("Original target names:", dataset.target_names)

# 打印本项目重新定义后的类别含义。
print("Project label mapping:", class_names)

## 3. 查看数据形状与前几条样本

In [ ]:
# 打印输入特征矩阵形状：[样本数, 特征数]。
print("X shape:", X.shape)

# 打印标签向量形状：[样本数]。
print("y shape:", y.shape)

# 打印样本数量。
print("Sample count:", X.shape[0])

# 打印特征数量。
print("Feature count:", X.shape[1])

# 显示前 5 条输入特征。
display(X.head())

# 显示前 5 个标签。
display(y.head())

## 4. 查看特征名称

这些特征来自细胞核图像的数值测量，并分为 mean、error、worst 等统计形式。

In [ ]:
# 把所有特征名称转换为列表。
feature_names = X.columns.tolist()

# 逐个打印特征序号和名称。
for feature_index, feature_name in enumerate(feature_names, start=1):
    # 使用两位宽度显示特征序号。
    print(f"{feature_index:02d}. {feature_name}")

## 5. 类别分布

类别分布会影响 Accuracy、Precision、Recall 和数据划分方式。

In [ ]:
# 统计每个标签的样本数量，并按标签顺序排列。
class_counts = y.value_counts().sort_index()

# 计算每个标签所占比例。
class_ratios = y.value_counts(normalize=True).sort_index()

# 把数量和比例合并成一个表格。
class_distribution = pd.DataFrame(
    {
        "class_name": [class_names[label] for label in class_counts.index],
        "count": class_counts.values,
        "ratio": class_ratios.values,
    },
    index=class_counts.index,
)

# 给索引命名。
class_distribution.index.name = "label"

# 显示类别分布表。
display(class_distribution)

In [ ]:
# 创建类别分布图画布。
figure, axis = plt.subplots(figsize=(6, 4))

# 绘制类别数量柱状图。
axis.bar(class_distribution["class_name"], class_distribution["count"])

# 设置横轴名称。
axis.set_xlabel("Class")

# 设置纵轴名称。
axis.set_ylabel("Sample Count")

# 设置标题。
axis.set_title("Class Distribution")

# 在每个柱子上方显示具体数量。
for bar in axis.patches:
    # 读取柱子的高度，也就是样本数量。
    height = bar.get_height()

    # 在柱子顶部写入数量。
    axis.text(
        bar.get_x() + bar.get_width() / 2,
        height,
        f"{int(height)}",
        ha="center",
        va="bottom",
    )

# 调整布局。
figure.tight_layout()

# 显示图片。
plt.show()

## 5.1 多数类基线与基准性能

新版课程强调：判断模型效果前，应先建立一个合理基准。

最简单的基线是始终预测样本数量最多的类别。它只提供最低参考，不代表业务可接受水平。


In [ ]:
# 找出样本数量最多的类别标签。
majority_label = int(y.value_counts().idxmax())

# 计算始终预测多数类时的 Accuracy。
majority_baseline_accuracy = float((y == majority_label).mean())

# 根据标签映射得到多数类名称。
majority_class_name = "malignant" if majority_label == 1 else "benign"

# 打印基线结果。
print("Majority class:", majority_class_name)
print("Majority-label baseline accuracy:", round(majority_baseline_accuracy, 4))

# 提醒：这个基线可能完全无法识别少数类，因此不能代替 Recall、F1 等指标。
print("This baseline is a minimum reference, not an acceptable final model.")


## 6. 检查缺失值与重复样本

缺失值、重复样本和异常数据会影响训练结果。

In [ ]:
# 统计每一列的缺失值数量。
missing_counts = X.isnull().sum()

# 统计整个数据集的缺失值总数。
total_missing = int(missing_counts.sum())

# 统计完全重复的特征行数量。
duplicate_count = int(X.duplicated().sum())

# 打印缺失值总数。
print("Total missing values:", total_missing)

# 打印重复样本数量。
print("Duplicate feature rows:", duplicate_count)

# 只显示存在缺失值的列；如果为空，说明没有缺失值。
display(missing_counts[missing_counts > 0])

## 7. 描述性统计

重点观察：

- mean：均值；
- std：标准差；
- min/max：范围；
- 25%/50%/75%：四分位数。

不同特征尺度差异很大，因此逻辑回归、SVM、MLP 需要标准化。

In [ ]:
# 计算全部特征的描述性统计。
descriptive_statistics = X.describe().transpose()

# 显示前 10 个特征的统计结果。
display(descriptive_statistics.head(10))

## 8. 查看特征尺度差异

In [ ]:
# 提取每个特征的最小值、最大值、均值和标准差。
scale_summary = descriptive_statistics[["min", "max", "mean", "std"]].copy()

# 按最大值从大到小排序，观察不同特征量级。
scale_summary = scale_summary.sort_values("max", ascending=False)

# 显示前 10 个量级最大的特征。
display(scale_summary.head(10))

## 9. 特征与标签的相关性

相关系数用于观察线性关系，但不能直接解释成因果关系。

In [ ]:
# 把特征与标签合并成一张表。
data_with_target = X.copy()

# 添加重新映射后的标签列。
data_with_target["is_malignant"] = y

# 计算每个特征与目标标签的 Pearson 相关系数。
target_correlations = (
    data_with_target.corr(numeric_only=True)["is_malignant"]
    .drop("is_malignant")
    .sort_values(key=lambda series: series.abs(), ascending=False)
)

# 显示绝对相关程度最高的 15 个特征。
display(target_correlations.head(15).to_frame("correlation_with_target"))

In [ ]:
# 选择与目标绝对相关性最高的 10 个特征。
top_features = target_correlations.head(10).index.tolist()

# 计算这 10 个特征之间的相关矩阵。
top_correlation_matrix = X[top_features].corr()

# 创建相关矩阵图画布。
figure, axis = plt.subplots(figsize=(10, 8))

# 把相关矩阵显示成图像。
image = axis.imshow(top_correlation_matrix, vmin=-1, vmax=1)

# 添加颜色条。
figure.colorbar(image, ax=axis)

# 设置横轴刻度位置。
axis.set_xticks(range(len(top_features)))

# 设置横轴特征名称。
axis.set_xticklabels(top_features, rotation=90)

# 设置纵轴刻度位置。
axis.set_yticks(range(len(top_features)))

# 设置纵轴特征名称。
axis.set_yticklabels(top_features)

# 设置标题。
axis.set_title("Correlation Matrix of Top Target-Correlated Features")

# 调整布局。
figure.tight_layout()

# 显示图片。
plt.show()

## 10. 分层划分训练集、验证集与测试集

目标比例：

```text
训练集：70%
验证集：15%
测试集：15%
```

`stratify=y` 用于让三个集合中的类别比例尽量接近原始数据。

In [ ]:
# 设置统一随机种子。
random_state = 42

# 第一次划分：70% 训练集，30% 临时集合。
X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=random_state,
    stratify=y,
)

# 第二次划分：把临时集合平均拆成验证集和测试集。
X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    random_state=random_state,
    stratify=y_temp,
)

# 打印三个集合的形状。
print("Train:", X_train.shape, y_train.shape)

# 打印验证集形状。
print("Validation:", X_val.shape, y_val.shape)

# 打印测试集形状。
print("Test:", X_test.shape, y_test.shape)

In [ ]:
# 创建空列表，用于保存三个集合的类别分布。
split_rows = []

# 依次处理训练、验证和测试标签。
for split_name, split_target in {
    "train": y_train,
    "validation": y_val,
    "test": y_test,
}.items():
    # 统计当前集合中的良性、恶性数量和恶性比例。
    split_rows.append(
        {
            "split": split_name,
            "sample_count": len(split_target),
            "benign_count": int((split_target == 0).sum()),
            "malignant_count": int((split_target == 1).sum()),
            "malignant_ratio": float((split_target == 1).mean()),
        }
    )

# 转换为 DataFrame。
split_summary = pd.DataFrame(split_rows)

# 显示数据划分摘要。
display(split_summary)

## 11. 为什么不能在这里标准化全部数据

错误做法：

```python
X_scaled = scaler.fit_transform(X)
```

然后再划分数据。

这样验证集和测试集会参与均值、标准差的计算，产生数据泄漏。

本项目在正式训练中使用 Pipeline：

```text
训练：scaler.fit(X_train) → transform(X_train) → model.fit(...)
验证：scaler.transform(X_val) → model.predict(...)
测试：scaler.transform(X_test) → model.predict(...)
```

验证集和测试集只使用训练集学到的标准化参数。

## 12. 本 Notebook 小结

完成本 Notebook 后，应能回答：

1. X 和 y 分别是什么？
2. 为什么本项目重新映射标签？
3. 数据集中有多少样本和特征？
4. 类别是否完全均衡？
5. 是否存在缺失值？
6. 为什么部分模型需要标准化？
7. 为什么要使用 `stratify=y`？
8. 训练集、验证集、测试集分别做什么？
9. 为什么测试集不能用于选择模型？
10. 为什么不能先对全部数据进行标准化？

## 13. 不同模型对特征缩放的需求不同

- Logistic Regression、SVM、MLP：通常需要 StandardScaler；
- Decision Tree、Random Forest、Gradient Boosting：通常不依赖特征距离或梯度尺度，不要求标准化；
- 是否标准化必须放入 Pipeline，并且只在训练数据上 fit。
